# Python `venv` Environments — The Isolation Awakens

*Alice gives Bob's two projects room to breathe.*

## Introduction — One User, Two Projects

> **Bob:** Your user install solved the problem, didn't it? Both projects can use the same packages.
>
> **Alice:** Until one needs a different version. Let's use Python's built-in `venv` module to give each project its own package directory.
>
> **Bob:** Another Python installation for every project?
>
> **Alice:** Not quite. Let's inspect what is shared and what is isolated.

Run this notebook in order in its [Dev Container](./README.md). It follows [The Raise of Python](../python_system_lab/python_system_lab.ipynb) and accompanies [Python `venv` Environments](../../docs/chapter-01/section-02.md). Cells run from `/workspace`; the notebook kernel remains in its own tooling environment throughout.

## Create Two Environments

> **Alice:** On Ubuntu the image includes `python3-venv`, which supplies the files needed to create environments. Use the base interpreter explicitly. This setup resets only the ignored `lab/` directory.
>
> **Bob:** So selecting another installed Python version at creation time selects that environment's base?
>
> **Alice:** Yes. `venv` does not install or manage Python versions itself.

In [ ]:
%%bash
set -euxo pipefail
rm -rf lab
mkdir -p lab/legacy lab/new
/usr/bin/python3 -m venv lab/legacy/.venv
/usr/bin/python3 -m venv lab/new/.venv
ls -la lab/legacy/.venv
ls -l lab/legacy/.venv/bin/python*
cat lab/legacy/.venv/pyvenv.cfg


## Inspect the Layout

> **Bob:** Those Python executables are symlinks to the base interpreter. Where is the isolation?
>
> **Alice:** The `pyvenv.cfg` beside the environment tells Python where its base is and whether to include system site-packages. `sys.prefix` points to the environment, but `sys.base_prefix` and the standard library still point to the base installation.

Compare the two interpreters and inspect their package locations. The default setting excludes system and user site-packages.

In [ ]:
%%bash
set -eu
for py in /usr/bin/python3 lab/legacy/.venv/bin/python; do
  "$py" -c 'import sys, sysconfig; print("executable:", sys.executable); print("prefix:", sys.prefix); print("base prefix:", sys.base_prefix); print("stdlib:", sysconfig.get_path("stdlib")); print("packages:", sysconfig.get_path("purelib")); print()'
done
lab/legacy/.venv/bin/python -m site


## Isolate Project Dependencies

> **Bob:** My older project needs `idna` 3.7, but yours needs 3.10. Won't the second install overwrite the first?
>
> **Alice:** Not when we run each environment's own pip. Let's install both versions and check the module paths.

These installs need network access. No package is installed into the system interpreter.

In [ ]:
%%bash
set -euxo pipefail
lab/legacy/.venv/bin/python -m pip install 'idna==3.7'
lab/new/.venv/bin/python -m pip install 'idna==3.10'
for py in lab/legacy/.venv/bin/python lab/new/.venv/bin/python; do
  "$py" -c 'import idna; print(idna.__version__, idna.__file__)'
done
lab/legacy/.venv/bin/python -c 'import idna; assert idna.__version__ == "3.7"'
lab/new/.venv/bin/python -c 'import idna; assert idna.__version__ == "3.10"'


## Activation and Import Resolution

> **Bob:** Do I have to activate an environment for it to work?
>
> **Alice:** No. Activation mainly puts its `bin` directory first on the shell's `PATH`; directly invoking its Python works too. Python then searches its own `sys.path`. Activation in one notebook shell cell does not persist in the next.

This cell demonstrates activation and deactivation in the same shell, while checking the selected package version.

In [ ]:
%%bash
set -eu
echo "Before: $(command -v python3)"
source lab/legacy/.venv/bin/activate
echo "Activated: $(command -v python3)"
python3 -c 'import idna, sys; print(idna.__version__, idna.__file__); print(*sys.path, sep="\n")'
deactivate
echo "After: $(command -v python3)"


## Epilogue — Disposable, Not Portable

> **Bob:** Two versions, same base interpreter, no conflict. I admit that is useful. Can I copy the environment to another machine?
>
> **Alice:** Recreate it instead. Scripts and configuration can contain absolute paths, and its base Python must still exist. Record project dependencies in a project definition or requirements file; `venv` itself does not lock them.
>
> **Bob:** I suppose one machine can have more than one environment after all.

The [technical companion](../../docs/chapter-01/section-02.md) covers the directory layout, platform differences, and the PyGuard project workflow.